In [ ]:
from pathlib import Path
import json
from collections import defaultdict, Counter

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from IPython.display import display

## SamplePairs

In [ ]:
dir_samplepairs = Path('/projects/mtg/projects/sample-identification/datasets/samplepairs_withnoise')

path_csv_samplepairs = dir_samplepairs / 'meta' / 'samples.csv'
path_tsv_samplepairs = dir_samplepairs / 'meta' / 'metadata.tsv'

In [ ]:
df_samplepairs = pd.read_csv(path_csv_samplepairs, delimiter=',')

with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_samplepairs)

In [ ]:
sources = set(df_samples['original_track_id'].str.strip())
receivers = set(df_samples['sample_track_id'].str.strip())
both = sources & receivers
print(both)

In [7]:
# Rows where the pair appears more than once (keeps all occurrences)
dupes = df_samples[
    df_samples.duplicated(subset=['original_track_id', 'sample_track_id'], keep=False)
].sort_values(['original_track_id', 'sample_track_id'])
display(dupes)

,sample_track_id,original_track_id


In [9]:
# Why are there duplicates?
edges = df_samples[['original_track_id', 'sample_track_id']].drop_duplicates()
print(len(edges))

102


In [ ]:
1258904

In [9]:
df_samplepairs_metadata = pd.read_csv(path_tsv_samplepairs, sep='\t', encoding='latin-1')
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(df_samplepairs_metadata)

,PairID,ID1,YTLink1,Artist1,Title1,Year1,Start1,ID2,YTLink2,Artist2,Title2,Year2,Start2,Website
0,1,1,https://www.youtube.com/watch?v=6jgBcsRCqI8,Berlioz,Something will happen,2024,0:20,2,https://www.youtube.com/watch?v=1dSRzxg7p1E,Willem Dafoe,Abandon perfection: try to fail,2020,0:15,https://www.whosampled.com/sample/1178057/Berlioz-Something-Will-Happen-Willem-Dafoe-Theoffcamerashow-Abandon-Perfection%3A-Try-to-Fail/
1,2,3,https://www.youtube.com/watch?v=N9bKBAA22Go,Future and Metro Boomin feat. Kendrick Lamar,Like that,2024,1:06,4,https://www.youtube.com/watch?v=kdf_GIV9Svo,Eazy-E,Eazy-Duz-It,1988,0:00,https://www.whosampled.com/sample/1158521/Future-Metro-Boomin-Kendrick-Lamar-Like-That-Eazy-E-Eazy-Duz-It/
2,3,5,https://www.youtube.com/watch?v=xeKSdJqFM6o,Jamie xx feat. Honey Dijon,Baddy on the Floor,2024,0:11,6,https://www.youtube.com/watch?v=UXBIrO-VlSM,Keni Burke,Let Somebody Love You,1981,0:01,https://www.whosampled.com/sample/1167203/Jamie-xx-Honey-Dijon-Baddy-on-the-Floor-Keni-Burke-Let-Somebody-Love-You/
3,4,7,https://www.youtube.com/watch?v=U8QhB8_0t2g,Benson Boone,Mystical Magical,2025,0:56,8,https://www.youtube.com/watch?v=vWz9VN40nCA,Olivia Newton-John,Physical,1981,0:46,https://www.whosampled.com/sample/1285693/Benson-Boone-Mystical-Magical-Olivia-Newton-John-Physical/
4,5,9,https://www.youtube.com/watch?v=h1AM48ZpSnw,Dennis DJ and Lusa Sonza,Motinha 2.0 (Mete Marcha),2025,0:30,10,https://www.youtube.com/watch?v=yU8IZ1gBwBU,MC Beth,Dana Da Motinha,2001,0:05,https://www.whosampled.com/sample/1258904/Dennis-DJ-Lu%C3%ADsa-Sonza-Motinha-2.0-(Mete-Marcha)-MC-Beth-Dan%C3%A7a-Da-Motinha/
5,6,11,https://www.youtube.com/watch?v=iD__IJWqwY8,YG Marley,Praise Jah in the Moonlight,2023,0:00,12,https://www.youtube.com/watch?v=a_O1kI6B9Bc,Bob Marley and The Wailers,Crisis,1978,0:16,https://www.whosampled.com/sample/1136450/YG-Marley-Praise-Jah-in-the-Moonlight-Bob-Marley-The-Wailers-Crisis/
6,7,13,https://www.youtube.com/watch?v=MEv4Hzf8Hhw,Morgan Wallen,Everything I Love,2023,0:28,14,https://www.youtube.com/watch?v=CoCaPJWqa28,The Allman Brothers Band,Midnight Rider,1970,0:24,https://www.whosampled.com/sample/1028230/Morgan-Wallen-Everything-I-Love-The-Allman-Brothers-Band-Midnight-Rider/
7,8,15,https://www.youtube.com/watch?v=421w1j87fEM,Burna Boy,Last Last,2022,0:09,16,https://www.youtube.com/watch?v=9_hKXk2qSuw,Toni Braxton,He Wasn't Man Enough,2000,0:14,https://www.whosampled.com/sample/942772/Burna-Boy-Last-Last-Toni-Braxton-He-Wasn%27t-Man-Enough/
8,9,17,https://www.youtube.com/watch?v=f36toA5VsoA,Playboi Carti,OPM BABI,2025,0:00,18,https://www.youtube.com/watch?v=hQOTdhuLBK8,Sample Magic,RSS2_75_vinyl_loop_holdon_F#maj,2022,0:00,https://www.whosampled.com/sample/1272964/Playboi-Carti-OPM-BABI-Sample-Magic-RSS2-75-vinyl-loop-holdon-F%23maj/
9,10,19,https://www.youtube.com/watch?v=r6glBvXN6H0,2Cellos,Sweet Child O'Mine,2021,0:03,20,https://www.youtube.com/watch?v=S6yuR8efotI,Johann Sebastian Bach,"Cello Suite No. 1 Prelude in G-Majeur, BWV 1007",1720,0:00,"https://www.whosampled.com/sample/918113/2Cellos-Sweet-Child-O%27-Mine-Johann-Sebastian-Bach-Cello-Suite-No.-1-Prelude-in-G-Majeur,-BWV-1007/"


In [20]:
sample100_whosampled_ids = set(df_samplepairs_metadata["Website"].str.split("/sample/").str[1].str.split("/").str[0].to_list())
print(len(sample100_whosampled_ids))

103


In [23]:
"1258904" in sample100_whosampled_ids

True

In [22]:
samplepairs_yt_ids = df_samplepairs_metadata["YTLink1"].to_list() + df_samplepairs_metadata["YTLink2"].to_list()

samplepairs_yt_ids = set(url.split("/watch?v=")[1] for url in samplepairs_yt_ids)
print(len(samplepairs_yt_ids))

206


In [10]:
import csv
from collections import defaultdict

result = defaultdict(list)

with open(path_csv_samples, newline="", encoding='latin-1') as f:
    reader = csv.DictReader(f)
    for row in reader:
        result[row["sample_track_id"]].append(row["original_track_id"])
result = dict(result)

In [11]:
len(result)

102

In [12]:
max([len(v) for v in result.values()])

1

In [13]:
result

{'T001': ['T002'],
 'T003': ['T004'],
 'T005': ['T006'],
 'T007': ['T008'],
 'T009': ['T010'],
 'T011': ['T012'],
 'T013': ['T014'],
 'T015': ['T016'],
 'T017': ['T018'],
 'T019': ['T020'],
 'T021': ['T022'],
 'T023': ['T024'],
 'T025': ['T026'],
 'T027': ['T028'],
 'T029': ['T030'],
 'T031': ['T032'],
 'T033': ['T034'],
 'T035': ['T036'],
 'T037': ['T038'],
 'T039': ['T040'],
 'T041': ['T042'],
 'T043': ['T044'],
 'T045': ['T046'],
 'T047': ['T048'],
 'T049': ['T050'],
 'T051': ['T052'],
 'T053': ['T054'],
 'T055': ['T056'],
 'T057': ['T058'],
 'T059': ['T060'],
 'T061': ['T062'],
 'T063': ['T064'],
 'T065': ['T066'],
 'T067': ['T068'],
 'T069': ['T070'],
 'T071': ['T072'],
 'T073': ['T074'],
 'T075': ['T076'],
 'T077': ['T078'],
 'T079': ['T080'],
 'T081': ['T082'],
 'T083': ['T084'],
 'T085': ['T086'],
 'T087': ['T088'],
 'T089': ['T090'],
 'T091': ['T092'],
 'T093': ['T094'],
 'T095': ['T096'],
 'T097': ['T098'],
 'T099': ['T100'],
 'T101': ['T102'],
 'T103': ['T104'],
 'T105': ['T